In [15]:
from kiteconnect import KiteConnect
import pandas as pd
import datetime as dt
import os

In [16]:
request_token=open("request_token.txt","r").read()
key_secrets=open("api_key.txt","r").read().split()
kite = KiteConnect(api_key=key_secrets[0])
data = kite.generate_session(request_token=request_token,api_secret=key_secrets[1])

In [17]:
# Get dump of all NSE instruments
instruments_dump=kite.instruments("NSE")
instruments_df=pd.DataFrame(instruments_dump)
instruments_df.to_csv("NSE_Instruments.csv",index=False)

In [18]:
def instrumentLookup(instruments_df,symbol):
    """Looks up instrument token for a given script from instrument dump"""
    try:
        return instruments_df[instruments_df.tradingsymbol==symbol].instrument_token.values[0]
    except:
        return -1


def fetchOHLC(ticker,interval,duration):
    """extracts historical data and outputs in the form of dataframe"""
    instrument = instrumentLookup(instruments_df,ticker)
    data = pd.DataFrame(kite.historical_data(instrument,dt.date.today()-dt.timedelta(duration), dt.date.today(),interval))
    data.set_index("date",inplace=True)
    return data

In [19]:
def bollBnd(DF,n):
    "function to calculate Bollinger Band"
    df = DF.copy()
    df["MA"] = df['close'].rolling(n).mean()
    #df["MA"] = df['close'].ewm(span=n,min_periods=n).mean()
    df["BB_up"] = df["MA"] + 2*df['close'].rolling(n).std(ddof=0) #ddof=0 is required since we want to take the standard deviation of the population and not sample
    df["BB_dn"] = df["MA"] - 2*df['close'].rolling(n).std(ddof=0) #ddof=0 is required since we want to take the standard deviation of the population and not sample
    df["BB_width"] = df["BB_up"] - df["BB_dn"]
    df.dropna(inplace=True)
    return df

In [20]:
ohlc = fetchOHLC("ICICIBANK","5minute",5)
bollBnd = bollBnd(ohlc,20)

In [21]:
ohlc

,open,high,low,close,volume
date,,,,,
2025-04-01 09:15:00+05:30,1340.00,1352.35,1339.55,1346.20,796452
2025-04-01 09:20:00+05:30,1345.90,1347.90,1344.05,1345.85,225771
2025-04-01 09:25:00+05:30,1345.90,1349.00,1341.65,1343.00,239011
2025-04-01 09:30:00+05:30,1342.40,1342.95,1337.00,1340.40,231862
2025-04-01 09:35:00+05:30,1340.65,1349.50,1338.80,1349.25,160504
...,...,...,...,...,...
2025-04-04 15:05:00+05:30,1334.50,1336.45,1334.40,1335.85,185687
2025-04-04 15:10:00+05:30,1335.70,1338.55,1335.10,1338.55,187375
2025-04-04 15:15:00+05:30,1338.40,1338.40,1335.00,1335.90,189060


In [23]:
bollBnd

,open,high,low,close,volume,MA,BB_up,BB_dn,BB_width
date,,,,,,,,,
2025-04-01 10:50:00+05:30,1330.00,1332.75,1328.75,1332.55,101838,1341.0550,1357.240886,1324.869114,32.371772
2025-04-01 10:55:00+05:30,1332.55,1333.00,1325.00,1325.20,154183,1340.0050,1357.399105,1322.610895,34.788211
2025-04-01 11:00:00+05:30,1325.05,1326.35,1324.35,1325.35,101358,1338.9800,1357.268614,1320.691386,36.577228
2025-04-01 11:05:00+05:30,1325.45,1327.90,1324.75,1326.95,78499,1338.1775,1357.088065,1319.266935,37.821130
2025-04-01 11:10:00+05:30,1326.70,1331.35,1326.50,1330.80,60370,1337.6975,1356.843922,1318.551078,38.292844
...,...,...,...,...,...,...,...,...,...
2025-04-04 15:05:00+05:30,1334.50,1336.45,1334.40,1335.85,185687,1334.0675,1336.514595,1331.620405,4.894190
2025-04-04 15:10:00+05:30,1335.70,1338.55,1335.10,1338.55,187375,1334.1950,1337.227309,1331.162691,6.064619
2025-04-04 15:15:00+05:30,1338.40,1338.40,1335.00,1335.90,189060,1334.1900,1337.210695,1331.169305,6.041391
